# Project 01: Predictive Analytics for Customer Churn
---

### Dataset Source:
Moro, S., Rita, P., & Cortez, P. (2014). Bank Marketing [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306.

### Dataset Description: 
This dataset contains customer and campaign information from direct phone marketing campaigns conducted between May 2008 and November 2010. The classification goal is to predict whether a contacted client will subscribe to a term deposit (`y`). The 10% “additional” sample (`bank-additional.csv`) contains 4,119 rows and 20 input features.

In [83]:
#Libraries
import pandas as pd

In [84]:
#Load the 10% additional dataset
df = pd.read_csv("bank-additional.csv", sep=';')

#Preview
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


## Exploratory Data Analysis (EDA)
---

In [85]:
#Basic Information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4119 non-null   object 
 2   marital         4119 non-null   object 
 3   education       4119 non-null   object 
 4   default         4119 non-null   object 
 5   housing         4119 non-null   object 
 6   loan            4119 non-null   object 
 7   contact         4119 non-null   object 
 8   month           4119 non-null   object 
 9   day_of_week     4119 non-null   object 
 10  duration        4119 non-null   int64  
 11  campaign        4119 non-null   int64  
 12  pdays           4119 non-null   int64  
 13  previous        4119 non-null   int64  
 14  poutcome        4119 non-null   object 
 15  emp.var.rate    4119 non-null   float64
 16  cons.price.idx  4119 non-null   float64
 17  cons.conf.idx   4119 non-null   f

In [86]:
#Statistical Summary
df.describe()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000
mean,40.113620,256.788055,2.537266,960.422190,0.190337,0.084972,93.579704,-40.499102,3.621356,5166.481695
std,10.313362,254.703736,2.568159,191.922786,0.541788,1.563114,0.579349,4.594578,1.733591,73.667904
min,18.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.635000,4963.600000
25%,32.000000,103.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.334000,5099.100000
50%,38.000000,181.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,317.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,88.000000,3643.000000,35.000000,999.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


In [87]:
#Check unique values (categorical columns)
for col in df.select_dtypes(include='object').columns:
    print(f"{col}: {df[col].unique()}")

job: ['blue-collar' 'services' 'admin.' 'entrepreneur' 'self-employed'
 'technician' 'management' 'student' 'retired' 'housemaid' 'unemployed'
 'unknown']
marital: ['married' 'single' 'divorced' 'unknown']
education: ['basic.9y' 'high.school' 'university.degree' 'professional.course'
 'basic.6y' 'basic.4y' 'unknown' 'illiterate']
default: ['no' 'unknown' 'yes']
housing: ['yes' 'no' 'unknown']
loan: ['no' 'unknown' 'yes']
contact: ['cellular' 'telephone']
month: ['may' 'jun' 'nov' 'sep' 'jul' 'aug' 'mar' 'oct' 'apr' 'dec']
day_of_week: ['fri' 'wed' 'mon' 'thu' 'tue']
poutcome: ['nonexistent' 'failure' 'success']
y: ['no' 'yes']


### EDA Obervations:
The numeric columns (`age`, `duration`, `campaign`, etc.) look mostly fine. `pdays` has 999 values, which likely means most clients were never previously contacted. Some of the categorical columns (`job`, `marital`, `education`, etc.) contain some `unknown` values. The target column (`y`) is clean with values `['no', 'yes']` (for modeling, they need to be encoded as 0/1). To address these two key issues, I plan to create a new binary indicator column called `pdays_never_contacted`, which will flag clients with `pdays = 999`. This ensures that our model can understand the effect of 'never previously contacted' while keeping the numeric feature clean. For handling the categorical `unknown` values, I plan to leave the `unknown` as its own category. When one-hot encoding the categorical variables for modeling, `unknown` will become a separate dummy variable. This preserves the info that the client status was unknown and allows the model to learn if unknown entries have predictive value. (Also, tree-based models like GBM or Random Forest can naturally split on `unknown` without imputation).

## Cleaning
---

In [88]:
#Create indicator for clients never contacted
df['pdays_never_contacted'] = (df['pdays'] == 999).astype(int)

#Replace 999 with NaN in pdays
df['pdays'] = df['pdays'].replace(999, pd.NA)

#Quick check
df[['pdays', 'pdays_never_contacted']].head(10)

,pdays,pdays_never_contacted
0,<NA>,1
1,<NA>,1
2,<NA>,1
3,<NA>,1
4,<NA>,1
5,<NA>,1
6,<NA>,1
7,<NA>,1
8,<NA>,1
9,<NA>,1


In [89]:
#Check how many unknown values exist
unknown_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan']

for col in unknown_cols:
    print(f"{col} - unknown count: {(df[col] == 'unknown').sum()}")

job - unknown count: 39
marital - unknown count: 11
education - unknown count: 167
default - unknown count: 803
housing - unknown count: 105
loan - unknown count: 105


## Encoding
---

In [90]:
#Encode target variable
y = df['y'].map({'yes': 1, 'no': 0})

#Drop target from features
X = df.drop(columns='y')

#Quick check
print(y.value_counts())
X.head(3)

y
0    3668
1     451
Name: count, dtype: int64


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,pdays_never_contacted
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,<NA>,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,1
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,<NA>,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,1
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,<NA>,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,1


In [91]:
#Identify categorical columns
cat_cols = X.select_dtypes(include='object').columns.tolist()

#Explicitly remove pdays if present (numeric campaign variable)
cat_cols = [col for col in cat_cols if col != 'pdays']

#Show
print("Categorical columns to encode:")
cat_cols

Categorical columns to encode:


['job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'poutcome']

In [92]:
#One-hot encode categorical features
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

#Check result
print("Shape after one-hot encoding:", X.shape)
X.head(3)

Shape after one-hot encoding: (4119, 54)


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
0,30,487,2,<NA>,0,-1.8,92.893,-46.2,1.313,5099.1,...,True,False,False,False,False,False,False,False,True,False
1,39,346,4,<NA>,0,1.1,93.994,-36.4,4.855,5191.0,...,True,False,False,False,False,False,False,False,True,False
2,25,227,1,<NA>,0,1.4,94.465,-41.8,4.962,5228.1,...,False,False,False,False,False,False,False,True,True,False


In [93]:
#Impute missing values and explicitly convert to numeric
X['pdays'] = pd.to_numeric(X['pdays'].fillna(0))

#Check
print("Total missing values:", X.isna().sum().sum())
print("pdays dtype:", X['pdays'].dtype)

Total missing values: 0
pdays dtype: int64


C:\Users\lynnm\AppData\Local\Temp\ipykernel_30892\1798118402.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X['pdays'] = pd.to_numeric(X['pdays'].fillna(0))


## Assess Data Quality
---

In [94]:
#Total missing values in the dataset
missing_total = X.isna().sum().sum()
print("Total missing values:", missing_total)

Total missing values: 0


In [95]:
#Verify all columns are numeric or boolean
print(X.dtypes.value_counts())

bool       43
int64       6
float64     5
Name: count, dtype: int64


In [96]:
#Distribution of target (class balance)
y.value_counts(normalize=True)

y
0    0.890507
1    0.109493
Name: proportion, dtype: float64

### Data Quality Assessment
Target Variable (`y`): ~89% 'no' and 11% 'yes' (indicates class imbalance)

Missing Values: None

Feature Types: 43 boolean, 6 integer, 5 float (all numeric or boolean for modeling)

Unusual Values: durations with large max values, `unknown` categories, and `pdays`: 999 → never contacted; imputed to 0 for modeling; indicator `pdays_never_contacted` added